# Session 12 lab: is the difference real?

Did the Lakeside Mart loyalty offer raise spend? You will answer it five ways: the formula, the bootstrap, the permutation test, the Welch t-test and a test of two proportions. You compute each number by hand with NumPy first, then check it against SciPy or statsmodels.

Rhythm for every step: **PREDICT**, **RUN**, **READ**, **CHANGE**. Cells with `# your code here` are yours to fill. The check cells say OK when your answer is right.

*Random seeds: use `np.random.default_rng(42)` exactly as written, and make the random calls in the order shown. Bootstrap and permutation checks allow a small tolerance.*

In [ ]:
# Run this cell first. It puts the course data files next to your notebook.
import os, re, zipfile, urllib.request
DATA_LINK = "https://drive.google.com/file/d/11Z5tr2iKuPROn5cyi9P7me6nd3naKfNp/view?usp=sharing"   # the Google Drive link to data.zip that your tutor shares (leave empty to upload data.zip yourself)
if not os.path.exists("orders.csv"):
    if DATA_LINK:
        m = re.search(r"/d/([A-Za-z0-9_-]+)|[?&]id=([A-Za-z0-9_-]+)", DATA_LINK)
        file_id = (m.group(1) or m.group(2)) if m else DATA_LINK
        urllib.request.urlretrieve("https://drive.google.com/uc?export=download&id=" + file_id, "data.zip")
    else:
        from google.colab import files
        print("Choose the file data.zip from your computer")
        uploaded = files.upload()
        os.replace(list(uploaded)[0], "data.zip")   # works even if the browser renamed it to data (1).zip
    zipfile.ZipFile("data.zip").extractall(".")
print("Data files ready:", sorted(f for f in os.listdir(".") if f.endswith((".csv", ".db"))))

In [ ]:
# Automatic check helper. Run this cell once. You do not need to read it.
_results = {}   # label -> True or False; running a check again replaces its old result
def check(label, value, expected, tol=0.01):
    try:
        import numpy as _np
        ok = bool(_np.all(_np.abs(_np.asarray(value, dtype=float) - _np.asarray(expected, dtype=float)) <= tol))
    except Exception:
        ok = False
    _results[label] = ok
    print(("OK       " if ok else "NOT YET  ") + label)
def show_checks():
    n = len(_results); k = sum(_results.values())
    print("All %d checks pass" % n if k == n and n else "%d of %d checks pass" % (k, n))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

---
## Step 1: Split the test into offer and control

Random assignment makes the two groups alike in everything except the offer (an A/B test).

**PREDICT first.** The offer group averaged 66,704 UGX and the control group about 61,660. How big a gap will you find, and is it big enough to trust?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
test = pd.read_csv('offer_test.csv')
test.head()

In [ ]:
# your code here: offer and control are NumPy arrays of next_month_spend_ugx for each group
offer = control = np.array([np.nan])
print(len(offer), len(control))
check('customers in offer', len(offer), 1245, 0)
check('customers in control', len(control), 1255, 0)

**READ the output.** Each record is one loyalty customer. Which group is the treatment group?

> My answer: 

**CHANGE one thing.** Print offer.mean() and control.mean(). Which one is larger, and by how much?

> What I saw: 

---
## Step 2: The lift, its standard error and the formula interval

Standard error of a mean = s / sqrt(n). The lift has a standard error that combines both groups. A 95 percent interval is the lift plus or minus 1.96 standard errors.

**PREDICT first.** Will the 95 percent interval for the lift include zero?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# your code here: lift = offer mean minus control mean; se = sqrt(var/n of each group, added); the interval is lift plus or minus 1.96 se (use ddof=1 for the variance)
lift = se = ci_low = ci_high = np.nan
print(np.round(lift), np.round(se), np.round(ci_low), np.round(ci_high))
check('lift (UGX)', lift, 5044.3348, 0.5)
check('standard error of the lift', se, 1525.8352, 1)
check('interval low end', ci_low, 2053.6977, 1)
check('interval high end', ci_high, 8034.9719, 1)

**READ the output.** Is zero inside the interval? What does that say about the lift?

> My answer: 

**CHANGE one thing.** Change 1.96 to 2.58. You now have a 99 percent interval. Is it wider or narrower? Why?

> What I saw: 

---
## Step 3: The bootstrap

Resample each group with replacement, at its own size, and recompute the lift. Repeat 5,000 times.

**PREDICT first.** Will the bootstrap interval be close to the formula interval, or very different?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# your code here: make rng once with seed 42; in a loop of 5000, resample offer then control with rng.choice and keep the difference of the means; then the std and the 2.5 and 97.5 percentiles
boot_se = boot_low = boot_high = np.nan
print(np.round(boot_se), np.round(boot_low), np.round(boot_high))
check('bootstrap standard error', boot_se, 1522.3772, 100)
check('bootstrap interval low end', boot_low, 1958.8800, 150)
check('bootstrap interval high end', boot_high, 8003.6757, 150)

**READ the output.** Compare boot_se with se from Step 2. Compare the two intervals.

> My answer: 

**CHANGE one thing.** Draw plt.hist(boot, bins=30) with a vertical line at the lift. Does the histogram look like a bell?

> What I saw: 

---
## Step 4: The permutation test

Under the null hypothesis the group labels carry no information. Shuffle them 5,000 times and count how often chance gives a gap at least as big as the lift.

**PREDICT first.** How many of 5,000 shuffled gaps will be as big as the real lift? Think about the histogram of shuffled gaps.

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# your code here: shuffle is_offer with rng.permutation 5000 times, keep the gap in mean spend each time; hits = how many gaps are at least as big as the lift (absolute values); p_perm = hits / 5000
hits = p_perm = np.nan
print(hits, p_perm)
check('shuffled gaps as big as the lift', hits, 7, 3)
check('permutation p-value', p_perm, 0.0014, 0.001)

**READ the output.** What does the p-value say? What does it NOT say?

> My answer: 

**CHANGE one thing.** Count only gaps at least as big as the lift in the positive direction. How does the p-value change? Which version is a one-sided test?

> What I saw: 

---
## Step 5: The Welch t-test and the test of two proportions

First by hand, then with SciPy and statsmodels. The numbers should agree.

**PREDICT first.** The t statistic is the lift divided by its standard error. Roughly what value do you expect, given the interval you found?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# your code here: t_hand = lift / se; res = stats.ttest_ind(offer, control, equal_var=False); take res.statistic and res.pvalue
t_hand = t_stat = p_t = np.nan
check('t statistic (SciPy)', t_stat, 3.3059, 0.001)
check('t by hand equals SciPy', t_hand - t_stat, 0, 0.001)
check('Welch p-value', p_t, 0.000960, 0.00005)

**READ the output.** Does the t-test p-value agree with the permutation p-value? Both are small. Which is more precise: p_t or p_perm?

> My answer: 

**PREDICT first.** Did more offer customers come back within 30 days (returned_within_30d)? Will the test find a real gap?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# your code here: counts of returners in offer then control; nobs = group sizes; proportions_ztest(counts, nobs)
z_stat = p_z = np.nan
check('z statistic', z_stat, 1.1040, 0.01)
check('p-value of the two-proportion test', p_z, 0.2696, 0.005)

**READ the output.** Is p_z below 0.05? Does the data show the offer brings customers back?

> My answer: 

**CHANGE one thing.** Change the proportion test to compare ONLY the first 500 customers of each group. What happens to the p-value, and why?

> What I saw: 

---
## Step 6: Effect size, and the decision note

A small p-value says the gap is real. The effect size says how big it is. Practical significance asks whether that size matters.

**PREDICT first.** Cohen's d is the lift divided by the pooled standard deviation. Will it be near 0.1, 0.5 or 1?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# your code here: the pooled standard deviation, then Cohen's d = lift / sd_pooled, and the lift as a percent of the control mean
cohens_d = lift_pct = np.nan
print(round(cohens_d, 3), round(lift_pct, 1))
check("Cohen's d", cohens_d, 0.1323, 0.005)
check('lift as percent of the control mean', lift_pct, 8.181, 0.05)

**READ the output.** Where does your d sit against the usual guide (0.2 small, 0.5 medium, 0.8 large)? Is it above the small mark?

> My answer: 

In [ ]:
note = 'Your turn: write three sentences for a manager. Give the lift, its interval, the p-value, the effect size, and one limit.'
print(note)

In [ ]:
check('decision note has numbers', any(ch.isdigit() for ch in note) and 'Your turn' not in note, 1)

**CHANGE one thing.** Suppose the offer costs 3,000 UGX per customer and the shop keeps 20 percent of each extra shilling spent (both hypothetical). Compute 0.20 * lift - 3000. Is it a gain or a loss? What about 0.20 * ci_high - 3000?

> What I saw: 

---
## Step 7: False positives: fake tests and many tests

Split the control group at random into two halves. The true gap is zero, so every p-value below 0.05 is a false positive (Type I error).

**PREDICT first.** Out of 100 such fake tests, how many will have p below 0.05?

Write your guess here before you run the next cell:

> My prediction: 

In [ ]:
# your code here: rng with seed 42; 100 times: permute the control indices, split at 627, Welch t-test the halves, count p below 0.05; family = chance of at least one false alarm in 20 tests
alarms = family = np.nan
check('false alarms in 100 fake tests', alarms, 5, 3)
check('chance of at least one false alarm in 20 tests', family, 0.6415, 0.001)
show_checks()

**READ the output.** About 5 of 100 fake tests raised an alarm. What happens to the chance of at least one false alarm when you run 20 tests? What would the Bonferroni cut-off be for 20 tests?

> My answer: 

**CHANGE one thing.** Extra, if time allows: peeking. Repeat the 100 fake tests, but look at the p-value after every batch of 100 customers in each half and stop at the first p below 0.05. Skeleton: for k in range(100, 628, 100): test a[:k] against b[:k]. Does the false alarm rate stay near 5 percent?

> What I saw: 

**You keep this notebook.** Save a copy to your Drive or download it as .ipynb. It holds the offer test done five ways and a decision note. Next session: regression.